# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and processing the [FAIR^2 Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
meta = dataset.metadata

print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")
print("Keywords:", getattr(meta, 'keywords', None))

## 2. Data Overview
Review the available record sets, fields, and their IDs defined in the Croissant schema. Each item is referenced by its `@id`.

In [ ]:
# List available record sets and their fields
print("Available record sets (by @id):\n")
record_sets = [rs for rs in meta.record_sets]
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print("  Fields and columns:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
        # List columns for the field if present
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"      Column @id: {col.id} | name: {col.name}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame. **All entity lookups always use their `@id`.**

In [ ]:
# For this dataset, extract records from main record set(s) by @id
record_sets = [rs.id for rs in meta.record_sets]
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display the columns of the principal data record set (choose the largest/most central one):
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Columns in main record set (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in schema.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (by its `@id`; see the `fields` printed above) for further filtering, normalization, and basic grouping operations.
<br />
**Remember:** All field/column/entity references use their `@id` as shown above.

In [ ]:
# For this EDA section, choose a numeric field by its `@id` (use field overview section to pick one)
numeric_field_id = None
group_field_id = None
if main_record_set_id:
    # Try to auto-detect a numeric field and a group field
    rs = [rs for rs in meta.record_sets if rs.id == main_record_set_id][0]
    for field in rs.fields:
        if getattr(field, 'data_type', '').lower() in ['float', 'number', 'integer']:
            numeric_field_id = field.id
            break
    for field in rs.fields:
        if getattr(field, 'data_type', '').lower() == 'text':
            group_field_id = field.id
            break

df = dataframes[main_record_set_id]
if numeric_field_id and numeric_field_id in df.columns:
    print(f"Using numeric field for analysis: {numeric_field_id}")
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    except Exception:
        pass
    threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as demo threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3g}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Could not find a numeric field by @id in DataFrame columns.")

## 5. Visualization
Let's chart the distribution of the chosen numeric field and show example relationships with grouping fields.
You may install `matplotlib` or `seaborn` if you wish to further explore visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field detected, or column not found in DataFrame.")

## 6. Conclusion
In this notebook, we've:
- Loaded Mass Spectrometry data described by a Croissant schema via `mlcroissant`.
- Inspected available record sets and fields by their `@id`.
- Loaded main records to DataFrame for analysis.
- Performed EDA by filtering, normalizing, and grouping on a numeric field, all referenced strictly by `@id`.
- Visualized the distributions to better understand dataset properties.

**Next steps** may include more domain-specific analysis, feature extraction, or advanced machine learning on the mass spectrometry data.